# Manga Speaker — Training Notebook v2

Run every cell top to bottom. Takes ~3-4 hours on T4.

**Before you start:**
- Runtime → Change Runtime Type → **T4 GPU**
- Upload your dataset zip to Google Drive at: `MyDrive/manga_speaker/`

**What changed from v1:**
- `rank=32` (was 16) — 2x model capacity
- `epochs=3` (was 2)
- Cosine LR warmup schedule
- Better prompts: model explicitly told not to repeat context
- Full deduplication in post-processing (not just prefix)

In [ ]:
# Cell 1: Check GPU
!nvidia-smi

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Install dependencies (~3 min first run)
!pip install -q transformers>=4.49 peft>=0.10 bitsandbytes>=0.43 accelerate>=0.28 qwen-vl-utils langgraph pillow pyyaml
print('Done')

In [ ]:
# Cell 4: Upload project code
# Zip your manga_speaker folder on your PC:
#   Right-click manga_speaker folder -> Send to -> Compressed (zipped) folder
# Then run this cell and upload that zip.

from google.colab import files
import os
print('Upload manga_speaker.zip (right-click your project folder -> compress it first)')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
!unzip -q "{zip_name}" -d /content/
# Handle both cases: zip contains folder or files directly
if os.path.exists('/content/manga_speaker'):
    os.chdir('/content/manga_speaker')
else:
    # If files are directly in /content/, rename
    folder = zip_name.replace('.zip', '')
    os.rename(f'/content/{folder}', '/content/manga_speaker')
    os.chdir('/content/manga_speaker')
print('Working dir:', os.getcwd())
!ls -la

In [ ]:
# Cell 5: Extract dataset from Google Drive
import os, glob

# Update this path if your zip has a different name
DRIVE_ZIP = '/content/drive/MyDrive/manga_speaker/BYTE ML Task Data-20261012T094429Z-1-001.zip'
DATASET_DIR = '/content/manga_speaker/dataset'

if not os.path.exists(f'{DATASET_DIR}/development/images'):
    print('Extracting dataset (~2 min)...')
    !unzip -q "{DRIVE_ZIP}" -d /content/manga_speaker/
    print('Extracted')
else:
    print('Already extracted')

imgs = len(glob.glob(f'{DATASET_DIR}/development/images/**/*.jpg', recursive=True))
print(f'Found {imgs} development images')

In [ ]:
# Cell 6: Verify data and show speaker distribution
import json
from collections import Counter

with open('splits.json') as f:
    splits = json.load(f)
with open('dataset/development/labels.jsonl') as f:
    labels = [json.loads(l) for l in f]

train_ids = set(splits['train'])
print(f'Train: {len(splits["train"])} seqs | Val: {len(splits["val"])} seqs | Labels: {len(labels)}')

spk_counter = Counter()
for seq in labels:
    if seq['sequence_id'] in train_ids:
        for page in seq['pages']:
            for line in page:
                spk_counter[line['speaker']] += 1

print('Speaker distribution in train split:')
for s, c in spk_counter.most_common(10):
    print(f'  {s:20s} {c}')

In [ ]:
# Cell 7: TRAIN — QLoRA fine-tuning (~3-4 hours)
#
# Settings explained:
#   --split all      = ALL 81 sequences (best accuracy)
#   --epochs 3       = 3 full data passes
#   --rank 32        = LoRA rank (was 16, doubled for capacity)
#   --lr 2e-4        = learning rate (cosine warmup built in)
#   --grad-accum 4   = gradient accumulation (effective batch size)
#   --load-4bit      = QLoRA 4-bit quantization (REQUIRED on T4)

!python scripts/train_lora.py \
    --dataset dataset \
    --labels dataset/development/labels.jsonl \
    --splits splits.json \
    --split all \
    --model-id Qwen/Qwen2.5-VL-3B-Instruct \
    --out adapters/qwen_manga_lora_v2 \
    --epochs 3 \
    --lr 2e-4 \
    --grad-accum 4 \
    --rank 32 \
    --context-pages 1 \
    --load-4bit \
    --seed 42

In [ ]:
# Cell 8: Save adapter to Google Drive
import shutil, os

DRIVE_SAVE = '/content/drive/MyDrive/manga_speaker/adapters_v2'
os.makedirs(DRIVE_SAVE, exist_ok=True)
shutil.copytree('adapters/qwen_manga_lora_v2', f'{DRIVE_SAVE}/qwen_manga_lora_v2', dirs_exist_ok=True)
print(f'Adapter saved to: {DRIVE_SAVE}')

In [ ]:
# Cell 9: Run inference on TEST set (for submission)
# This uses the brand new v2 adapter

!python scripts/run_inference.py \
    --dataset dataset \
    --ids-from dataset/sample_submission.jsonl \
    --out test_predictions_v2.jsonl \
    --config configs/default.yaml \
    --adapter adapters/qwen_manga_lora_v2 \
    --set model.load_4bit=true model.model_id=Qwen/Qwen2.5-VL-3B-Instruct

print('Done')

In [ ]:
# Cell 10 (OPTIONAL): Score on validation set to compare v1 vs v2

!python scripts/run_inference.py \
    --dataset dataset \
    --ids-from splits.json \
    --split val \
    --out val_predictions_v2.jsonl \
    --config configs/default.yaml \
    --adapter adapters/qwen_manga_lora_v2 \
    --set model.load_4bit=true model.model_id=Qwen/Qwen2.5-VL-3B-Instruct

!python dataset/score.py \
    --predictions val_predictions_v2.jsonl \
    --labels dataset/development/labels.jsonl \
    --output scores_v2.json \
    --split val \
    --splits splits.json

import json
with open('scores_v2.json') as f:
    sc = json.load(f)
m = sc['macro']
print('=== v2 Validation Scores ===')
print(f"  balanced_joint_f1:   {m['balanced_joint_f1']:.4f}  (v1 was 0.1523)")
print(f"  speaker_accuracy:    {m['speaker_accuracy_on_matched']:.4f}  (v1 was 0.5704)")
print(f"  text_order_score:    {m['text_order_score']:.4f}  (v1 was 0.5865)")
print(f"  joint_f1:            {m['joint_f1']:.4f}  (v1 was 0.3761)")

In [ ]:
# Cell 11: Download everything
import shutil
from google.colab import files

# Copy to Drive
shutil.copy('test_predictions_v2.jsonl',
            '/content/drive/MyDrive/manga_speaker/test_predictions_v2.jsonl')

# Download directly to PC
files.download('test_predictions_v2.jsonl')
print('Done — check your Downloads folder')